<a href="https://colab.research.google.com/github/KinhoLeung/WhisperJAV/blob/main/notebook/WhisperJAV_kaggle_parallel_edition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://kaggle.com/kernels/welcome?src=https://github.com/KinhoLeung/WhisperJAV/blob/main/notebook/WhisperJAV_kaggle_parallel_edition.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/></a>

# WhisperJAV Kaggle Parallel Edition (v1.8.5)

**Universal Parallel Workflow**

| Platform | Logic | Storage |
|----------|-------|---------|
| **Kaggle** | **Parallel** (2x T4) | **Input**: `/kaggle/input` (Dataset) <br> **Output**: `/kaggle/working` (Artifacts) |
| **Colab** | **Sequential** (1x GPU) | **Input/Output**: Google Drive |

---
### **Workflow**
1. **Configure**: Select your settings, models, and audio preferences.
2. **Setup**: Installs dependencies and prepares the environment (**Run Once**).
3. **Transcribe**: Processes your video files using the configured settings.
4. **Translate**: (Optional) Uses AI to translate subtitles to English.

<small><i>Tip: Select your Dataset in Kaggle, then use "Run All". The notebook is designed to Fail Fast if resources are missing.</i></small>

> **Note on Speech Enhancer**: The speech enhancer option is only available when using the built-in ensemble mode (Colab Expert notebook). In this parallel notebook, the enhancer UI fields are informational only and do not affect processing. This limitation will be resolved in v1.8.6.</small>

---
<sub>If you find WhisperJAV useful, consider supporting the project: <a href="https://buymeacoffee.com/meizhong">buymeacoffee.com/meizhong</a></sub>

In [ ]:
#@title Step 1: Expert Configuration { display-mode: "form" }

#@markdown ## 📁 Files & Output
folder_name = "WhisperJAV" #@param {type:"string"}
subtitle_language = "Japanese" #@param ["Japanese", "English (auto-translate)", "English (AI translate)"]
spoken_language = "Japanese" #@param ["Japanese", "Chinese", "English", "Korean"]

#@markdown ---
#@markdown ## 1️⃣ Pass 1 Configuration (GPU 0)
pass1_quality = "balanced" #@param ["faster", "fast", "balanced", "fidelity", "transformers"]
pass1_sensitivity = "aggressive" #@param ["conservative", "balanced", "aggressive"]
pass1_model = "large-v2" #@param ["automatic", "large-v2", "large-v3", "turbo", "kotoba-bilingual", "kotoba-v2.0", "kotoba-v2.1", "kotoba-v2.2"]

#@markdown **Expert Audio Setup (Pass 1)**
pass1_scene_detector = "semantic" #@param ["automatic", "auditok", "silero", "semantic"]
pass1_speech_segmenter = "ten" #@param ["automatic", "silero", "ten", "none"]
pass1_speech_enhancer = "none" #@param ["none", "ffmpeg-dsp", "clearvoice", "zipenhancer", "bs-roformer"]
#@markdown <font size="1">auditok=energy (fast), silero=VAD, semantic=texture (complex audio) | enhancer: ffmpeg-dsp(no GPU), clearvoice(48k), bs-roformer(vocal)</font>

#@markdown **FFmpeg Filters (Pass 1)** *(only if enhancer is ffmpeg-dsp)*
pass1_ffmpeg_amplify = True #@param {type:"boolean"}
pass1_ffmpeg_loudnorm = False #@param {type:"boolean"}
pass1_ffmpeg_compress = False #@param {type:"boolean"}
pass1_ffmpeg_highpass = False #@param {type:"boolean"}

#@markdown ---
#@markdown ## 2️⃣ Pass 2 Configuration (GPU 1)
pass2_quality = "fidelity" #@param ["faster", "fast", "balanced", "fidelity", "transformers"]
pass2_sensitivity = "balanced" #@param ["conservative", "balanced", "aggressive"]
pass2_model = "turbo" #@param ["automatic", "large-v2", "large-v3", "turbo", "kotoba-bilingual", "kotoba-v2.0", "kotoba-v2.1", "kotoba-v2.2"]

#@markdown **Expert Audio Setup (Pass 2)**
pass2_scene_detector = "semantic" #@param ["automatic", "auditok", "silero", "semantic"]
pass2_speech_segmenter = "ten" #@param ["automatic", "silero", "ten", "none"]
pass2_speech_enhancer = "ffmpeg-dsp" #@param ["none", "ffmpeg-dsp", "clearvoice", "zipenhancer", "bs-roformer"]

#@markdown **FFmpeg Filters (Pass 2)** *(only if enhancer is ffmpeg-dsp)*
pass2_ffmpeg_amplify = True #@param {type:"boolean"}
pass2_ffmpeg_loudnorm = False #@param {type:"boolean"}
pass2_ffmpeg_compress = False #@param {type:"boolean"}
pass2_ffmpeg_highpass = False #@param {type:"boolean"}

#@markdown ---
#@markdown ## 🔗 Merge Strategy
merge_method = "prefer first pass" #@param ["automatic", "keep all", "prefer first pass", "prefer second pass"]

#@markdown ---
#@markdown ## 🤖 AI Translation *(if selected)*
translation_service = "local" #@param ["local", "deepseek", "openrouter", "gemini", "claude", "gpt", "glm", "groq"]
local_model = "gemma-9b" #@param ["gemma-9b", "llama-8b", "llama-3b", "auto"]
#@markdown <font size="1">local: Free, runs on GPU. gemma-9b (8GB+ VRAM), llama-8b (6GB+), llama-3b (3GB+). Cloud providers require API key.</font>
api_key = "" #@param {type:"string"}
translation_style = "standard" #@param ["standard", "explicit"]

#@markdown ---
#@markdown ## ⚙️ Session
opening_credit = "" #@param {type:"string"}
closing_credit = "Subs by WhisperJAV" #@param {type:"string"}
auto_disconnect = True #@param {type:"boolean"}

# ═══════════════════════════════════════════
# CONFIGURATION LOGIC
# ═══════════════════════════════════════════
import sys
import os

# Detect Platform Early
if "google.colab" in sys.modules:
    PLATFORM = "colab"
elif os.path.exists("/kaggle"):
    PLATFORM = "kaggle"
else:
    PLATFORM = "local"

# WhisperJAV commands — both Colab and Kaggle use direct system install (no venv)
if PLATFORM == "colab":
    WHISPERJAV_CMD = "whisperjav"
    WHISPERJAV_TRANSLATE_CMD = "whisperjav-translate"
else:
    # Kaggle: use explicit module command
    WHISPERJAV_CMD = f"{sys.executable} -m whisperjav.main"
    WHISPERJAV_TRANSLATE_CMD = f"{sys.executable} -m whisperjav.translate.cli"

# Mapping dictionaries
combine_map = {"automatic": "smart_merge", "keep all": "full_merge",
               "prefer first pass": "pass1_primary", "prefer second pass": "pass2_primary"}
language_map = {"Japanese": "native", "English (auto-translate)": "direct-to-english",
                "English (AI translate)": "llm"}
tone_map = {"standard": "standard", "explicit": "pornify"}
spoken_language_map = {"Japanese": "japanese", "Chinese": "chinese", "English": "english", "Korean": "korean"}

# Model mapping (None = use pipeline default)
model_map = {
    "automatic": None,
    "large-v2": "large-v2",
    "large-v3": "large-v3",
    "turbo": "turbo",
    "kotoba-bilingual": "kotoba-tech/kotoba-whisper-bilingual-v1.0",
    "kotoba-v2.0": "kotoba-tech/kotoba-whisper-v2.0",
    "kotoba-v2.1": "kotoba-tech/kotoba-whisper-v2.1",
    "kotoba-v2.2": "kotoba-tech/kotoba-whisper-v2.2"
}

# Define model compatibility:
KOTOBA_MODELS = {"kotoba-bilingual", "kotoba-v2.0", "kotoba-v2.1", "kotoba-v2.2"}
LEGACY_PIPELINES = {"faster", "fast", "balanced", "fidelity"}

# Auto-correct incompatible model-pipeline combinations
warnings_list = []

# Check Pass 1 compatibility
if pass1_model in KOTOBA_MODELS and pass1_quality in LEGACY_PIPELINES:
    warnings_list.append(f"Pass 1: {pass1_model} requires 'transformers' pipeline. Auto-correcting from '{pass1_quality}' to 'transformers'.")
    pass1_quality = "transformers"

# Check Pass 2 compatibility
if pass2_model in KOTOBA_MODELS and pass2_quality in LEGACY_PIPELINES:
    warnings_list.append(f"Pass 2: {pass2_model} requires 'transformers' pipeline. Auto-correcting from '{pass2_quality}' to 'transformers'.")
    pass2_quality = "transformers"

# Memory warning
heavy_enhancers = {'clearvoice', 'bs-roformer', 'zipenhancer'}
if pass1_speech_enhancer in heavy_enhancers and pass2_speech_enhancer in heavy_enhancers:
    warnings_list.append("Using GPU-based enhancement on both passes may cause OOM on T4 GPU (Sequential Mode). Suggest using ffmpeg-dsp for one pass.")

# Helpers
def build_ffmpeg_filters(amplify, loudnorm, compress, highpass):
    """Combine selected FFmpeg filters into comma-separated string."""
    filters = []
    if amplify: filters.append("amplify")
    if loudnorm: filters.append("loudnorm")
    if compress: filters.append("compress")
    if highpass: filters.append("highpass")
    return ",".join(filters) if filters else None

def map_value(val):
    return None if val == "automatic" else val

def map_segmenter(val):
    return "none" if val == "none" else map_value(val)

# Unified Config Construction
WHISPERJAV_CONFIG = {
    'pass1_pipeline': pass1_quality,
    'pass1_sensitivity': pass1_sensitivity,
    'pass1_speech_segmenter': map_segmenter(pass1_speech_segmenter),
    'pass1_model': model_map[pass1_model],
    # Expert fields merged directly
    'pass1_scene_detector': map_value(pass1_scene_detector),
    'pass1_speech_enhancer': None if pass1_speech_enhancer == "none" else pass1_speech_enhancer,
    'pass1_ffmpeg_filters': build_ffmpeg_filters(pass1_ffmpeg_amplify, pass1_ffmpeg_loudnorm, pass1_ffmpeg_compress, pass1_ffmpeg_highpass) if pass1_speech_enhancer == "ffmpeg-dsp" else None,
    
    'pass2_pipeline': pass2_quality,
    'pass2_sensitivity': pass2_sensitivity,
    'pass2_speech_segmenter': map_segmenter(pass2_speech_segmenter),
    'pass2_model': model_map[pass2_model],
    # Expert fields merged directly
    'pass2_scene_detector': map_value(pass2_scene_detector),
    'pass2_speech_enhancer': None if pass2_speech_enhancer == "none" else pass2_speech_enhancer,
    'pass2_ffmpeg_filters': build_ffmpeg_filters(pass2_ffmpeg_amplify, pass2_ffmpeg_loudnorm, pass2_ffmpeg_compress, pass2_ffmpeg_highpass) if pass2_speech_enhancer == "ffmpeg-dsp" else None,
    
    'merge_strategy': combine_map[merge_method],
    'folder_name': folder_name,
    'subtitle_language': language_map[subtitle_language],
    'language': spoken_language_map[spoken_language],
    'translation_service': translation_service,
    'local_model': local_model,
    'api_key': api_key,
    'translation_style': tone_map[translation_style],
    'opening_credit': opening_credit,
    'closing_credit': closing_credit,
    'auto_disconnect': auto_disconnect,
    # Platform info
    'platform': PLATFORM,
    'whisperjav_cmd': WHISPERJAV_CMD,
    'whisperjav_translate_cmd': WHISPERJAV_TRANSLATE_CMD,
    # Compatibility checks for Step 2
    '_pass1_quality': pass1_quality,
    '_pass1_sensitivity': pass1_sensitivity,
    '_pass1_speech_segmenter': pass1_speech_segmenter,
    '_pass1_model': pass1_model,
    '_pass2_quality': pass2_quality,
    '_pass2_sensitivity': pass2_sensitivity,
    '_pass2_speech_segmenter': pass2_speech_segmenter,
    '_pass2_model': pass2_model,
    '_merge_method': merge_method,
    '_subtitle_language': subtitle_language,
    '_translation_style': translation_style,
}

from IPython.display import display, HTML

# Display warnings
for warning in warnings_list:
    display(HTML(f'<div style="padding:6px 10px;background:#fef9c3;border-radius:4px;font-size:10px;margin-bottom:4px"><b>⚠️ Auto-corrected:</b> {warning}</div>'))

# Build status display
p1_info = f"{pass1_quality}"
if pass1_speech_segmenter != "automatic":
    p1_info += f"/{pass1_speech_segmenter}"
if pass1_model != "automatic":
    p1_info += f"/{pass1_model}"
if pass1_scene_detector != "automatic":
    p1_info += f"/{pass1_scene_detector}"

p2_info = f"{pass2_quality}"
if pass2_speech_segmenter != "automatic":
    p2_info += f"/{pass2_speech_segmenter}"
if pass2_model != "automatic":
    p2_info += f"/{pass2_model}"
if pass2_scene_detector != "automatic":
    p2_info += f"/{pass2_scene_detector}"

display(HTML(f'<div style="padding:10px;background:#e0f2fe;border-radius:4px;font-size:11px">'
             f'<b>Parallel Configuration Loaded</b><br>'
             f'Platform: {PLATFORM.upper()} | Pass 1: {p1_info} | Pass 2: {p2_info}<br>'
             f'Merge: {merge_method} | Folder: {folder_name} | Language: {spoken_language}'
             f'</div>'))

In [ ]:
#@title Step 2: Setup & Environment (Run Once) { display-mode: "form" }
#@markdown - **Fails Fast** if GPU or Internet is missing.
#@markdown - **Colab**: Uses `install_colab.sh` (installs into system Python)
#@markdown - **Kaggle**: Direct pip install with pinned versions (numpy 2.x compatible)

import os
import shutil
import subprocess
import sys
import time
import socket
from pathlib import Path

# Fix matplotlib backend conflict
os.environ['MPLBACKEND'] = 'Agg'
os.environ['TORCH_HUB_TRUST_REPO'] = '1'  # Fix #253

def section(title):
    print(f"\n{'---'*14}\n{title}\n{'---'*14}")

def status(msg, ok=True):
    print(f"{'OK' if ok else 'FAIL'} {msg}")

def must(condition, msg):
    if not condition:
        raise RuntimeError(f"SETUP FAILED: {msg}")

# ═══════════════════════════════════════════
# 1. PRE-FLIGHT CHECKS (Fail Fast)
# ═══════════════════════════════════════════
section("PRE-FLIGHT CHECKS")

# Get config from Step 1
if "WHISPERJAV_CONFIG" not in globals():
    from IPython.display import display, HTML
    display(HTML('<div style="background:#fee2e2;padding:10px;border-radius:4px;"><b>Run Step 1 First</b></div>'))
    raise RuntimeError("Step 1 required")

cfg = WHISPERJAV_CONFIG
PLATFORM = cfg['platform']
print(f"Platform: {PLATFORM.upper()}")
print(f"Python:   {sys.executable} ({sys.version.split()[0]})")

# Python version check - WhisperJAV requires Python 3.10-3.12
py_version = sys.version_info
if py_version >= (3, 13):
    from IPython.display import display, HTML
    display(HTML('<div style="background:#fee2e2;padding:10px;border-radius:4px;"><b>Python 3.13+ Not Supported</b><br>WhisperJAV requires Python 3.10-3.12 (openai-whisper incompatible with 3.13+)</div>'))
    raise RuntimeError(f"Python {sys.version.split()[0]} not supported. Requires 3.10-3.12.")
elif py_version < (3, 10):
    raise RuntimeError(f"Python {sys.version.split()[0]} too old. Requires 3.10-3.12.")
status(f"Python version OK ({sys.version.split()[0]})")

# Check GPU
gpu_check = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True)
must(gpu_check.returncode == 0 and bool(gpu_check.stdout.strip()), "No GPU detected. Switch runtime to GPU.")
gpus = [line.strip() for line in gpu_check.stdout.splitlines() if line.strip()]
status(f"GPU(s) Detected: {len(gpus)} ({', '.join(gpus)})")

# Check Internet
def check_internet():
    endpoints = [("www.baidu.com", 80), ("www.google.com", 80), ("1.1.1.1", 53)]
    for host, port in endpoints:
        try:
            socket.create_connection((host, port), timeout=3.0).close()
            return True
        except OSError:
            continue
    return False

must(check_internet(), "Internet disabled/unreachable. Cannot install dependencies.")
status("Internet reachable")

# ═══════════════════════════════════════════
# 2. INSTALLATION (Platform-specific)
# ═══════════════════════════════════════════
section("INSTALLING DEPENDENCIES")
install_start = time.time()

if PLATFORM == "colab":
    # ═══════════════════════════════════════════
    # COLAB: Use install_colab.sh (installs into system Python)
    # Reuses Colab's pre-installed PyTorch and CUDA stack
    # ═══════════════════════════════════════════
    REPO_URL = "https://github.com/meizhong986/WhisperJAV.git"
    REPO_PATH = "/content/WhisperJAV"
    SCRIPT_PATH = f"{REPO_PATH}/installer/install_colab.sh"
    
    def run_installer():
        """Run install script with real-time output streaming."""
        env = {**os.environ, "PATH": f"{os.environ.get('PATH', '')}:{os.path.expanduser('~/.local/bin')}"}
        process = subprocess.Popen(
            ["bash", SCRIPT_PATH],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
            text=True,
            env=env
        )
        for line in iter(process.stdout.readline, ''):
            print(line, end='', flush=True)
        process.wait()
        return process.returncode
    
    # Check if already installed
    check = subprocess.run(["python3", "-c", "import whisperjav"], capture_output=True)
    if check.returncode == 0:
        status("WhisperJAV already installed (skipping)")
    else:
        # Fresh install
        print("Installing WhisperJAV (uv-accelerated)...\n")
        sys.stdout.flush()
        if not os.path.exists(REPO_PATH):
            print(f"Cloning {REPO_URL}...")
            result = subprocess.run(["git", "clone", REPO_URL, REPO_PATH], capture_output=True, text=True)
            must(result.returncode == 0, f"Clone failed: {result.stderr}")

        must(os.path.exists(SCRIPT_PATH), f"Install script not found at {SCRIPT_PATH}")
        returncode = run_installer()
        must(returncode == 0, "Installation failed")
    
    status(f"Installation Complete ({time.time() - install_start:.0f}s)")

else:
    # ═══════════════════════════════════════════
    # KAGGLE: Direct pip install with numpy 2.x compatible versions
    # Kaggle pre-loads numpy 2.x which is now the correct version
    # ═══════════════════════════════════════════
    
    # ─────────────────────────────────────────
    # KAGGLE ENVIRONMENT SETUP
    # ─────────────────────────────────────────
    # Set up cache directories to avoid RAM disk overflow
    kaggle_cache = Path("/kaggle/working/.cache")
    kaggle_cache.mkdir(parents=True, exist_ok=True)
    kaggle_temp = Path("/kaggle/working/temp")
    kaggle_temp.mkdir(parents=True, exist_ok=True)
    
    # Set environment variables for HuggingFace and temp files
    os.environ["HF_HOME"] = str(kaggle_cache)
    os.environ["TRANSFORMERS_CACHE"] = str(kaggle_cache / "transformers")
    os.environ["TORCH_HOME"] = str(kaggle_cache / "torch")
    os.environ["TMPDIR"] = str(kaggle_temp)
    os.environ["TEMP"] = str(kaggle_temp)
    os.environ["TMP"] = str(kaggle_temp)
    status(f"Cache directories configured: {kaggle_cache}")
    
    # ─────────────────────────────────────────
    # KAGGLE INSTALLATION (numpy 2.x)
    # ─────────────────────────────────────────
    
    def run_shell(name, cmd):
        """Run a shell command with timeout."""
        print(f"... installing {name}")
        try:
            r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
        except subprocess.TimeoutExpired:
            raise RuntimeError(f"Install timed out (10m): {name}")
        if r.returncode != 0:
            print((r.stderr or r.stdout or "")[-2000:])
            raise RuntimeError(f"Install failed: {name}")
        status(f"Installed {name}")

    def run_pip(name, packages, no_deps=False):
        """Run pip install with timeout."""
        print(f"... installing {name}")
        cmd = [sys.executable, "-m", "pip", "install", "-q"]
        if no_deps:
            cmd.append("--no-deps")
        cmd.extend(packages)
        try:
            r = subprocess.run(cmd, check=False, text=True, capture_output=True, timeout=600)
        except subprocess.TimeoutExpired:
            raise RuntimeError(f"Pip Install timed out (10m): {name}")
        if r.returncode != 0:
            print(f"--- pip stderr for {name} ---")
            print((r.stderr or r.stdout or "")[-2000:])
            raise RuntimeError(f"Pip Install failed: {name}")
        status(f"Installed {name}")

    # Determine sudo usage
    prefix = ""
    if shutil.which("sudo") and not (hasattr(os, "geteuid") and os.geteuid() == 0):
        prefix = "sudo "

    # A. System Layer (APT)
    sys_pkgs = "ffmpeg portaudio19-dev libc++1 libc++abi1"
    run_shell("System Tools (apt)", f"{prefix}apt-get update -qq && {prefix}apt-get install -y -qq {sys_pkgs}")
    must(shutil.which("ffmpeg"), "FFmpeg installation verified")

    # B. Core Acceleration Layer (Pip)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=False)

    core_libs = [
        "tqdm", "numba>=0.60.0", "llvmlite>=0.46.0", "tiktoken", "soundfile", "auditok", "requests", "colorama", "regex",
        "numpy>=2.0.0", "scipy>=1.13.0", "librosa>=0.11.0",
        "pysrt", "srt", "aiofiles", "jsonschema", "Pillow", "pyloudnorm", "pydantic>=2,<3",
        "faster-whisper>=1.1.0", "transformers", "optimum", "accelerate", "huggingface-hub>=0.25.0",
        "ten-vad", "silero-vad>=6.0", "pydub",
        "modelscope>=1.20", "onnxruntime>=1.16.0", "addict", "simplejson", "sortedcontainers", "packaging"
    ]
    run_pip("Core Acceleration Libs", core_libs)

    # C. Application Layer (Git) - Install from main branch for latest fixes
    git_pkgs = [
        ("ffmpeg-python", "git+https://github.com/kkroening/ffmpeg-python.git"),
        ("Whisper", "git+https://github.com/openai/whisper.git@main"),
        ("Stable-TS", "git+https://github.com/meizhong986/stable-ts-fix-setup.git@main"),
        ("ClearVoice", "git+https://github.com/modelscope/ClearerVoice-Studio.git#subdirectory=clearvoice"),
        ("WhisperJAV", "git+https://github.com/meizhong986/WhisperJAV.git@main"),
    ]

    for name, url in git_pkgs:
        run_pip(name, [url], no_deps=False)

    # D. Force reinstall pinned versions (ensure consistency after git installs)
    run_pip("Ensure Core Consistency", [
        "--force-reinstall", "--upgrade",
        "numpy>=2.0.0", "scipy>=1.13.0", "librosa>=0.11.0",
        "numba>=0.60.0", "llvmlite>=0.46.0", "scikit-learn>=1.4.0"
    ], no_deps=True)

    status(f"Installation Complete ({time.time() - install_start:.0f}s)")

# ═══════════════════════════════════════════
# 3. VERIFICATION
# ═══════════════════════════════════════════
section("VERIFICATION")

# Determine which python to use for verification
verify_python = sys.executable

# Verify Import
result = subprocess.run([verify_python, "-c", "import whisperjav; print(whisperjav.__version__)"], capture_output=True, text=True)
if result.returncode == 0:
    status(f"WhisperJAV {result.stdout.strip()} Ready")
else:
    raise RuntimeError(f"Installation successful but import failed: {result.stderr}")

# Verify TEN VAD
result = subprocess.run([verify_python, "-c", "import ten_vad"], capture_output=True, text=True)
if result.returncode == 0:
    status("TEN VAD Ready")
else:
    status("TEN VAD Warning (Import failed - Silero fallback)", False)

# Verify numpy version
result = subprocess.run([verify_python, "-c", "import numpy; print(numpy.__version__)"], capture_output=True, text=True)
if result.returncode == 0:
    numpy_version = result.stdout.strip()
    if numpy_version.startswith("1."):
        status(f"WARNING: NumPy {numpy_version} detected - WhisperJAV requires numpy 2.x", False)
    else:
        status(f"NumPy {numpy_version} (compatible)")

# Stamp Success
WHISPERJAV_SETUP_COMPLETE = {
    "timestamp": time.time(),
    "platform": PLATFORM,
    "gpu_count": len(gpus)
}
print("\nEnvironment Ready. Go to Step 3.")

In [ ]:
#@title Step 3: Execution (Transcribe) { display-mode: "form" }
#@markdown - Auto-detects **Files** (Kaggle Dataset or Colab Drive).
#@markdown - Auto-detects **Parallel** (2x GPU) or **Sequential** mode.
#@markdown - **Optimized**: Uses GPUs to process DIFFERENT videos simultaneously.

import os
import shutil
import sys
import time
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import HTML, FileLink, display

# ═══════════════════════════════════════════
# PLATFORM SETUP
# ═══════════════════════════════════════════

if "WHISPERJAV_CONFIG" not in globals():
    raise RuntimeError("Run Step 1 first!")

cfg = WHISPERJAV_CONFIG
PLATFORM = cfg['platform']

# Define Paths based on Platform
if PLATFORM == "kaggle":
    INPUT_DIR = Path("/kaggle/input")
    OUTPUT_DIR = Path("/kaggle/working/output")
    TEMP_DIR = Path("/tmp/whisperjav")
    try:
        DATASET_DIR = next(d for d in INPUT_DIR.iterdir() if d.is_dir())
    except StopIteration:
        DATASET_DIR = INPUT_DIR
elif PLATFORM == "colab":
    INPUT_DIR = Path(f"/content/drive/MyDrive/{cfg['folder_name']}")
    OUTPUT_DIR = INPUT_DIR  
    TEMP_DIR = Path("/content/temp")
    DATASET_DIR = INPUT_DIR
else:
    INPUT_DIR = Path("./")
    OUTPUT_DIR = Path("./output")
    TEMP_DIR = Path("./temp")
    DATASET_DIR = INPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)
WHISPERJAV_OUTPUT_DIR = OUTPUT_DIR

# ═══════════════════════════════════════════
# FILE SCANNING
# ═══════════════════════════════════════════
print(f"Scanning {DATASET_DIR}...")
VIDEO_EXTS = {'.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv', '.webm', '.m4v', '.mp3', '.wav', '.flac', '.m4a'}
videos = sorted([f for f in DATASET_DIR.rglob("*") if f.suffix.lower() in VIDEO_EXTS])

if not videos:
    print(f"No media files found in {DATASET_DIR}")
else:
    print(f"Found {len(videos)} files")

# ═══════════════════════════════════════════
# GPU SETUP
# ═══════════════════════════════════════════
import torch
ngpus = torch.cuda.device_count()
print(f"GPUs Available: {ngpus}")

if ngpus >= 2:
    MODE = "PARALLEL"
else:
    MODE = "SEQUENTIAL"
print(f"Mode: {MODE}")

# ═══════════════════════════════════════════
# EXECUTION ENGINE
# ═══════════════════════════════════════════

def build_cmd(video_path, output_dir):
    """Build CLI command for Pass 1 configuration."""
    pass_num = 1 # Always use Pass 1 config for maximum simplicity
    pass_temp = TEMP_DIR / f"work_{video_path.stem}"
    pass_temp.mkdir(parents=True, exist_ok=True)
    
    # Base command
    executable = cfg.get('whisperjav_cmd', sys.executable + " -m whisperjav.main")
    if str(executable).endswith('whisperjav'):
        cmd = [str(executable)]
    else:
        cmd = list(str(executable).split())

    cmd += [
        str(video_path),
        "--output-dir", str(output_dir),
        "--temp-dir", str(pass_temp),
        "--mode", cfg['pass1_pipeline'],
        "--sensitivity", cfg['pass1_sensitivity'],
        "--subs-language", "direct-to-english" if cfg['subtitle_language'] == 'direct-to-english' else "native",
    ]

    model = cfg['pass1_model']
    if model: cmd += ["--model", model]

    seg = cfg.get('pass1_speech_segmenter')
    if seg and seg != 'none': cmd += ["--speech-segmenter", seg]

    scene = cfg.get('pass1_scene_detector')
    if scene: cmd += ["--scene-detection-method", scene]

    return cmd

def process_video_on_gpu(video, gpu_id, video_index, total_videos):
    """Run transcription for a single video on a specific GPU."""
    print(f"\n[GPU {gpu_id}] Starting ({video_index}/{total_videos}): {video.name}")
    
    # We use a subfolder inside output_dir to avoid filename collisions during processing
    task_out = OUTPUT_DIR / f"gpu{gpu_id}_{video_index}"
    task_out.mkdir(parents=True, exist_ok=True)
    
    cmd = build_cmd(video, task_out)
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    
    t0 = time.time()
    proc = None
    try:
        # Standard timeout 4 hours per video
        proc = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=14400)
        success = proc.returncode == 0
    except Exception as e:
        print(f"  [GPU {gpu_id}] Execution error: {e}")
        success = False
        
    found_srts = list(task_out.glob(f"{video.stem}*.srt"))
    if success and found_srts:
        final_file = OUTPUT_DIR / f"{video.stem}.whisperjav.srt"
        shutil.copy(found_srts[0], final_file)
        # Cleanup task folder
        shutil.rmtree(task_out, ignore_errors=True)
        print(f"  [GPU {gpu_id}] OK ({time.time()-t0:.0f}s) -> {final_file.name}")
        return final_file
    else:
        log = proc.stderr[-500:] if proc and proc.stderr else "No output"
        print(f"  [GPU {gpu_id}] FAILED. Log tail: {log}")
        return None

# ═══════════════════════════════════════════
# MAIN LOOP: VIDEO PARALLELISM
# ═══════════════════════════════════════════

WHISPERJAV_RESULTS = []

if MODE == "PARALLEL" and len(videos) > 1:
    print(f"Dual GPUs detected. Processing DIFFERENT videos on each GPU...\n")
    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = []
        for i, video in enumerate(videos):
            assigned_gpu = str(i % 2)
            futures.append(executor.submit(process_video_on_gpu, video, assigned_gpu, i + 1, len(videos)))
        
        for future in as_completed(futures):
            res = future.result()
            if res: WHISPERJAV_RESULTS.append(res)
else:
    print(f"Sequential mode (Single GPU or single video).")
    for i, video in enumerate(videos):
        res = process_video_on_gpu(video, "0", i + 1, len(videos))
        if res: WHISPERJAV_RESULTS.append(res)

print("\nDONE!")
if PLATFORM == "kaggle":
    shutil.make_archive("/kaggle/working/subtitles", 'zip', OUTPUT_DIR)
    print("Output zipped: subtitles.zip")


In [ ]:
#@title Step 4: AI Translation (Optional) { display-mode: "form" }
#@markdown ## Provider Settings
translation_provider = "local" #@param ["local", "deepseek", "openrouter", "gemini", "claude", "gpt", "glm", "groq"]
local_model = "gemma-9b" #@param ["gemma-9b", "llama-8b", "llama-3b", "auto"]
#@markdown <font size="1">local: Free, runs on GPU. gemma-9b (8GB+ VRAM), llama-8b (6GB+), llama-3b (3GB+). Cloud providers require API key.</font>
api_key = "" #@param {type:"string"}
model_override = "" #@param {type:"string"}

#@markdown ## Translation Settings
target_language = "english" #@param ["english", "chinese", "spanish", "indonesian"]
source_language = "japanese" #@param ["japanese", "korean", "chinese"]
tone = "pornify" #@param ["standard", "pornify"]

#@markdown ## Context (Optional)
#@markdown *Leave blank for batch processing of different movies.*
movie_title = "" #@param {type:"string"}
actress = "" #@param {type:"string"}
movie_plot = "" #@param {type:"string"}

#@markdown ## Advanced
scene_threshold = 60 #@param {type:"integer"}
batch_size = 30 #@param {type:"integer"}

import os
import sys
import subprocess
import time
from pathlib import Path
from IPython.display import HTML, display, FileLink

def status(msg, ok=True):
    print(f"{'OK' if ok else 'FAIL'} {msg}")

# ═══════════════════════════════════════════
# 1. INPUT VALIDATION & DISCOVERY
# ═══════════════════════════════════════════
# Get config from Step 1 for venv paths
cfg = globals().get("WHISPERJAV_CONFIG", {})

# Use OUTPUT_DIR from Step 3 if available, otherwise reconstruct
if "WHISPERJAV_OUTPUT_DIR" not in globals():
    if cfg:
        fname = cfg["folder_name"]
        if "google.colab" in sys.modules:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            WHISPERJAV_OUTPUT_DIR = Path(f"/content/drive/MyDrive/{fname}")
        elif os.path.exists("/kaggle"):
            WHISPERJAV_OUTPUT_DIR = Path("/kaggle/working/output")
        else:
            WHISPERJAV_OUTPUT_DIR = Path(f"{fname}_output").absolute()
    else:
        # Fallback for manual run without Step 1
        WHISPERJAV_OUTPUT_DIR = Path(".").resolve()

# Recover SRTs — check Step 3 results first, then scan output dir
targets = []
if "WHISPERJAV_RESULTS" in globals() and WHISPERJAV_RESULTS:
    targets = WHISPERJAV_RESULTS
elif WHISPERJAV_OUTPUT_DIR.exists():
    print(f"Scanning for SRTs in: {WHISPERJAV_OUTPUT_DIR}")
    # Find all source SRTs (excluding existing translations and intermediate pass files)
    candidates = sorted(list(WHISPERJAV_OUTPUT_DIR.glob("*.srt")))
    targets = [t for t in candidates
               if not t.name.endswith(f".{target_language}.srt")
               and '.pass1.' not in t.name and '.pass2.' not in t.name]

if not targets:
    display(HTML('<div style="background:#fee2e2;padding:10px;border-radius:4px;"><b>No Subtitles Found</b><br>Run Step 3, or ensure Output directory has SRTs.</div>'))
    raise RuntimeError("No inputs")

# Check Metadata safety
if len(targets) > 1 and (movie_title or actress or movie_plot):
    display(HTML(f'<div style="background:#fff7ed;padding:10px;border-radius:4px;border:1px solid #fdba74"><b>Metadata Warning</b><br>'
                 f'You are applying the same Movie Title/Plot/Actress to <b>{len(targets)} different files</b>.<br>'
                 f'If these are different movies, please clear the Context fields above.</div>'))

# ═══════════════════════════════════════════
# 2. CREDENTIALS
# ═══════════════════════════════════════════
is_local = translation_provider == 'local'
final_key = api_key

if not is_local:
    if not final_key:
        # Try Step 1 Config (if matches provider)
        if cfg.get("api_key") and cfg.get("translation_service") == translation_provider:
            final_key = cfg["api_key"]
        # Try Kaggle Secrets
        elif os.path.exists("/kaggle"):
            from kaggle_secrets import UserSecretsClient
            try:
                final_key = UserSecretsClient().get_secret(f"{translation_provider.upper()}_API_KEY")
            except: pass

    if not final_key:
        raise RuntimeError(f"Missing API Key for {translation_provider}. Enter it in the form or use 'local' provider for free GPU translation.")

# ═══════════════════════════════════════════
# 3. INTERACTIVE CONFIRMATION
# ═══════════════════════════════════════════
print(f"Ready to translate {len(targets)} files.")
if is_local:
    print(f"Provider: local ({local_model})")
    print("Note: First run downloads model (~5GB) and llama-cpp-python (~700MB)")
else:
    print(f"Provider: {translation_provider}")
print(f"Target: {target_language} | Tone: {tone}")
if model_override:
    print(f"Model: {model_override}")

# Skip interactive prompt on Kaggle and Colab (batch mode)
if PLATFORM not in ("kaggle", "colab"):
    try:
        input("\nPress [Enter] to start translation (or Stop cell to cancel)...")
    except (EOFError, KeyboardInterrupt):
        print("Cancelled by user.")
        sys.exit(0)

# ═══════════════════════════════════════════
# 4. EXECUTION
# ═══════════════════════════════════════════
print(f"\nStarting Translation Service...")
translated_files = []
start_time = time.time()

# Determine which command to use: venv binary for Colab, sys.executable for Kaggle/local
translate_cmd = cfg.get('whisperjav_translate_cmd')

for srt in targets:
    print(f"\nProcessing: {srt.name}")
    
    # Use venv binary for Colab, sys.executable for Kaggle/local
    if translate_cmd:
        cmd = [translate_cmd]
    else:
        cmd = [sys.executable, "-m", "whisperjav.translate.cli"]
    
    cmd.extend([
        "-i", str(srt),
        "--provider", translation_provider,
        "--source", source_language,
        "--target", target_language,
        "--tone", tone,
        "--scene-threshold", str(scene_threshold),
        "--max-batch-size", str(batch_size),
        "--stream" # Stream progress to stderr
    ])
    
    # Add API key for cloud providers
    if not is_local:
        cmd.extend(["--api-key", final_key])
    
    # Model selection
    if is_local:
        cmd.extend(["--model", local_model])
    elif model_override:
        cmd.extend(["--model", model_override])
    
    # Optional Args
    if movie_title: cmd.extend(["--movie-title", movie_title])
    if actress: cmd.extend(["--actress", actress])
    if movie_plot: cmd.extend(["--movie-plot", movie_plot])
    
    # Run with real-time streaming
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1, # Line buffered
            encoding='utf-8',
            errors='replace' # Prevent decoding errors
        )
        
        # Read stderr (progress) and stdout (result path)
        output_path = None
        while True:
            # Check stderr for progress updates
            err_line = process.stderr.readline()
            if err_line:
                print(err_line.strip(), file=sys.stderr)
            
            # Check stdout for final filename
            out_line = process.stdout.readline()
            if out_line:
                line = out_line.strip()
                if line and line.endswith('.srt'):
                    output_path = line
                print(line)

            # Break safely
            if not err_line and not out_line and process.poll() is not None:
                break
                
        if process.returncode == 0 and output_path:
            out_file = Path(output_path)
            if out_file.exists():
                status(f"Completed: {out_file.name}")
                translated_files.append(out_file)
            else:
                 status(f"Error: Output file not found: {output_path}", False)
        else:
            status("Translation failed (check logs above)", False)

    except Exception as e:
        status(f"Exception: {e}", False)

# ═══════════════════════════════════════════
# 5. PACKAGING & DOWNLOAD
# ═══════════════════════════════════════════
if translated_files:
    base_out = WHISPERJAV_OUTPUT_DIR if "WHISPERJAV_OUTPUT_DIR" in globals() else translated_files[0].parent
    zip_path = base_out / f"translated_{translation_provider}_{int(time.time())}.zip"
    
    import zipfile
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for t in translated_files:
            zf.write(t, t.name)
            
    print(f"\nTranslations Bundle: {zip_path.name}")
    
    if os.path.exists("/kaggle"):
         display(FileLink(str(zip_path.relative_to(Path.cwd())), result_html_prefix="<b>Download Translations: </b>"))
    elif "google.colab" in sys.modules:
        try:
            from google.colab import files
            files.download(str(zip_path))
        except: pass
else:
    print("\nNo translations produced.")

print(f"\nTotal Time: {(time.time() - start_time)/60:.1f} min")